# XGBoost

## Data Preparation and Preprocessing

In [5]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

# getting rid of " " in TotalCharges
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)# make it numeric, if cannot put NaN(this is what errors="coerce" does)

#we saw that when Total Charges " " tenure=0 meaning new customers so we can
#make TotalCharges=0 for them
df["TotalCharges"] = df["TotalCharges"].fillna(0) #replace NaNs with 0s

df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

# drop meaningless feature and target to create X
X = df.drop(columns=["Churn", "customerID"])
# choosing Churn as the target
y = df["Churn"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [6]:
categorical_features = [cname for cname in X.columns if
                    X[cname].dtype == "object"]

# Select numerical columns
numerical_features = [cname for cname in X.columns if 
                X[cname].dtype in ['int64', 'float64']]

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    remainder="passthrough" #don't touch numerical features
)

## Baseline XGBoost

In [11]:
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

xgb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        random_state=0,
        eval_metric="logloss"
    ))
])

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
y_prob = xgb_model.predict_proba(X_test)[:, 1]

### Evaluation

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7778566359119943
Precision: 0.5867507886435331
Recall: 0.5054347826086957
F1-score: 0.5430656934306569
ROC-AUC: 0.8117051434657312


In [13]:
print("Train accuracy:", xgb_model.score(X_train, y_train))
print("Test accuracy:", xgb_model.score(X_test, y_test))

Train accuracy: 0.940539581114661
Test accuracy: 0.7778566359119943


#### a light overfitting maybe.. and Roc-auc is almost same with the result of tuned-DT model

## Hyperparameter Tuning

In [14]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [2, 3, 4, 5, 6],
    "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1, 0.2],
    "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
    "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0]
}

In [15]:
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_distributions,
    n_iter=30,
    cv=5,
    scoring="roc_auc",
    random_state=0,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_)

Best parameters: {'classifier__subsample': 0.8, 'classifier__n_estimators': 200, 'classifier__max_depth': 3, 'classifier__learning_rate': 0.03, 'classifier__colsample_bytree': 0.8}
Best CV ROC-AUC: 0.8527139665948267


### Tuned XGBoost Model

In [16]:
best_xgb = random_search.best_estimator_

In [17]:
y_pred_tuned = best_xgb.predict(X_test)
y_prob_tuned = best_xgb.predict_proba(X_test)[:, 1]

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned))
print("Recall:", recall_score(y_test, y_pred_tuned))
print("F1-score:", f1_score(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_tuned))

Accuracy: 0.794889992902768
Precision: 0.6338983050847458
Recall: 0.5081521739130435
F1-score: 0.5641025641025641
ROC-AUC: 0.8312750595163514


## Model comparison

In [18]:
comparison = pd.DataFrame({
    "XGBoost": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
        roc_auc_score(y_test, y_prob)
    ],
    "Tuned XGBoost": [
        accuracy_score(y_test, y_pred_tuned),
        precision_score(y_test, y_pred_tuned),
        recall_score(y_test, y_pred_tuned),
        f1_score(y_test, y_pred_tuned),
        roc_auc_score(y_test, y_prob_tuned)
    ]
}, index=[
    "Accuracy",
    "Precision",
    "Recall",
    "F1-score",
    "ROC-AUC"
])

comparison.round(4)

,XGBoost,Tuned XGBoost
Accuracy,0.7779,0.7949
Precision,0.5868,0.6339
Recall,0.5054,0.5082
F1-score,0.5431,0.5641
ROC-AUC,0.8117,0.8313


## Conclusion

#### The baseline XGBoost model achieved a ROC-AUC of approximately 0.81, with a moderate gap between training and test accuracy indicating some overfitting. Hyperparameter tuning improved the model's generalization performance, increasing the test ROC-AUC from 0.81 to 0.83.

#### The tuned XGBoost model achieved the highest ROC-AUC among the models evaluated so far, slightly outperforming Logistic Regression. However, Logistic Regression still achieved slightly higher accuracy, precision, recall, and F1-score at the default classification threshold of 0.5.

#### This shows that a higher ROC-AUC does not necessarily translate into better performance at a specific classification threshold. The results also demonstrate that a more complex ensemble model does not automatically outperform a simpler linear model across every evaluation metric.